# 1. Gemini-3-Flash-Preview (dual-bfws-ffparser)

In [2]:
# Cell 1: Setup imports, paths, and Gemini key (new google.genai SDK)
from pathlib import Path
import os
import re
import json
import time
import subprocess
from datetime import datetime

from dotenv import load_dotenv
from google import genai

# Project root (run notebook from formalizer-scaling/src/pddl_gen)
PROJECT_ROOT = Path.cwd().parents[1]

DATA_DIR = PROJECT_ROOT / "data"
PROMPT_PATH = PROJECT_ROOT / "prompts" / "only_pddl_instruction.txt"
DOMAIN_PDDL_PATH = DATA_DIR / "pddl" / "domain.pddl"
DESC_DIR = DATA_DIR / "descriptions"

OUTPUT_ROOT = PROJECT_ROOT / "output" / "direct_pddl" / "gemini_3_flash"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

VAL_PATH = Path("/home/sagemaker-user/tools/VAL/build/linux64/Release/bin/Validate")

# Load .env and configure Gemini client
load_dotenv(PROJECT_ROOT / ".env")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
if not GOOGLE_API_KEY:
  raise ValueError("GOOGLE_API_KEY not found in .env")

client = genai.Client(api_key=GOOGLE_API_KEY)

# Gemini 3 Flash model code (preview)
MODEL_NAME = "gemini-3-flash-preview"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("VAL exists:", VAL_PATH.exists())
print("Domain PDDL exists:", DOMAIN_PDDL_PATH.exists())
print("Prompt exists:", PROMPT_PATH.exists())
print("Model:", MODEL_NAME)

PROJECT_ROOT: /home/sagemaker-user/formalizer-scaling
OUTPUT_ROOT: /home/sagemaker-user/formalizer-scaling/output/direct_pddl/gemini_3_flash
VAL exists: True
Domain PDDL exists: True
Prompt exists: True
Model: gemini-3-flash-preview


In [3]:
# Cell 2: Define target problems and load static prompt/domain text
PROBLEM_IDS = [f"p{i:03d}" for i in range(101, 201)]  # p101..p200

PROMPT_PREFIX = PROMPT_PATH.read_text()
DOMAIN_PDDL_TEXT = DOMAIN_PDDL_PATH.read_text()

print("Num problems:", len(PROBLEM_IDS))
print("First/last:", PROBLEM_IDS[0], PROBLEM_IDS[-1])
print("Prompt chars:", len(PROMPT_PREFIX))
print("Domain PDDL chars:", len(DOMAIN_PDDL_TEXT))

Num problems: 100
First/last: p101 p200
Prompt chars: 2544
Domain PDDL chars: 997


In [4]:
# Cell 3: Load NL descriptions for one problem ID
def load_problem_descriptions(problem_id: str):
  domain_path = DESC_DIR / f"{problem_id}_domain.txt"
  problem_path = DESC_DIR / f"{problem_id}_problem.txt"

  if not domain_path.exists():
      raise FileNotFoundError(f"Missing domain description: {domain_path}")
  if not problem_path.exists():
      raise FileNotFoundError(f"Missing problem description: {problem_path}")

  return domain_path.read_text(), problem_path.read_text()

In [5]:
# Cell 4: Build prompt for one problem (combined primer + 3-input setup)
def build_prompt(problem_id: str):
  nl_domain, nl_problem = load_problem_descriptions(problem_id)

  return (
      f"{PROMPT_PREFIX}\n\n"
      "You are an expert PDDL engineer.\n"
      "Create a syntactically correct and semantically consistent problem file "
      "using the information below.\n\n"
      "You are given three inputs:\n"
      "1) Original natural language domain description\n"
      "2) Original natural language problem description\n"
      "3) Ground-truth domain PDDL (fixed reference)\n\n"
      "### Original domain description\n"
      f"{nl_domain}\n\n"
      "### Original problem description\n"
      f"{nl_problem}\n\n"
      "### Ground-truth domain PDDL\n"
      f"{DOMAIN_PDDL_TEXT}\n\n"
      "Output contract:\n"
      "- Generate ONLY the final problem PDDL.\n"
      "- Wrap it in exactly one <problem_file>...</problem_file> block.\n"
      "- Do not output any <domain_file> block.\n"
      "- Do not include any extra text before or after <problem_file>...</problem_file>."
  )

In [45]:
# Cell 5: Call Gemini for one prompt
def call_gemini(prompt: str, temperature: float = 0.4, max_output_tokens: int = 50000):
  response = client.models.generate_content(
      model=MODEL_NAME,
      contents=prompt,
      config={
          "temperature": temperature,
          "max_output_tokens": max_output_tokens,
      },
  )
  return (response.text or "").strip()

In [7]:
# Cell 6: Parse model output and extract problem_file block
def extract_problem_pddl(raw_text: str):
  if not raw_text:
      return ""

  matches = re.findall(r"<problem_file>(.*?)</problem_file>", raw_text, flags=re.DOTALL | re.IGNORECASE)
  if matches:
      return matches[-1].strip()

  return ""

In [8]:
# Cell 7: Solve one generated problem via planning.domains API (online solver)
import requests

def run_solver(domain_pddl_path: Path, problem_pddl_path: Path, work_dir: Path, timeout_sec: int = 300):
  solver_name = "dual-bfws-ffparser"
  base_url = "https://solver.planning.domains:5001"

  domain_txt = domain_pddl_path.read_text()
  problem_txt = problem_pddl_path.read_text()

  print(f"[solver-online] submit: {solver_name}")
  try:
      submit = requests.post(
          f"{base_url}/package/{solver_name}/solve",
          json={"domain": domain_txt, "problem": problem_txt},
          timeout=300,
      )
      submit.raise_for_status()
      job = submit.json()
  except Exception as e:
      return {
          "plan_found": False,
          "plan_text": "",
          "solver_error": f"submit failed: {e}",
          "stdout": "",
          "stderr": "",
      }

  result_path = job.get("result")
  if not result_path:
      return {
          "plan_found": False,
          "plan_text": "",
          "solver_error": f"unexpected submit response: {job}",
          "stdout": "",
          "stderr": "",
      }

  result_url = f"{base_url}{result_path}"
  start = time.time()

  while True:
      if time.time() - start > timeout_sec:
          return {
              "plan_found": False,
              "plan_text": "",
              "solver_error": f"timeout after {timeout_sec}s",
              "stdout": "",
              "stderr": "",
          }

      try:
          poll = requests.post(result_url, timeout=30)
          poll.raise_for_status()
          res = poll.json()
      except Exception as e:
          return {
              "plan_found": False,
              "plan_text": "",
              "solver_error": f"poll failed: {e}",
              "stdout": "",
              "stderr": "",
          }

      status = res.get("status")
      if status == "PENDING":
          print("[solver-online] pending...")
          time.sleep(0.5)
          continue
      break

  # Handle explicit service-side error
  if "Error" in res:
      return {
          "plan_found": False,
          "plan_text": "",
          "solver_error": str(res.get("Error")),
          "stdout": "",
          "stderr": "",
      }

  result = res.get("result", {})
  stdout = result.get("stdout", "") if isinstance(result, dict) else ""
  stderr = result.get("stderr", "") if isinstance(result, dict) else ""
  output = result.get("output", {}) if isinstance(result, dict) else {}

  # Extract plan text from planning.domains format variants
  plan_text = ""
  if isinstance(output, dict):
      plan_text = output.get("plan", "") or ""
  elif isinstance(output, str):
      plan_text = output

  if isinstance(plan_text, list):
      plan_text = "\n".join(str(x) for x in plan_text)

  plan_text = str(plan_text).strip()

  if plan_text:
      # Write plan file in the same place expected by Cell 12
      (work_dir / "plan").write_text(plan_text + ("\n" if not plan_text.endswith("\n") else ""))
      print("[solver-online] plan found.")
      return {
          "plan_found": True,
          "plan_text": plan_text,
          "solver_error": "",
          "stdout": stdout,
          "stderr": stderr,
      }

  err = (stderr + "\n" + stdout).strip() or "plan not found"
  print("[solver-online] no plan.")
  return {
      "plan_found": False,
      "plan_text": "",
      "solver_error": err,
      "stdout": stdout,
      "stderr": stderr,
  }

In [9]:
# Cell 8: Validate plan with VAL (live logs, no timeout)
import select

def run_val(domain_pddl_path: Path, problem_pddl_path: Path, plan_path: Path):
  if not VAL_PATH.exists():
      return {
          "val_ok": False,
          "val_output": f"VAL executable not found: {VAL_PATH}",
      }

  cmd = [str(VAL_PATH), "-v", str(domain_pddl_path), str(problem_pddl_path), str(plan_path)]
  print(f"[val] start: {' '.join(cmd)}")

  proc = subprocess.Popen(
      cmd,
      stdout=subprocess.PIPE,
      stderr=subprocess.PIPE,
      text=True,
      bufsize=1,
  )

  out_lines = []
  err_lines = []

  # Live-stream until process exits
  while True:
      if proc.poll() is not None:
          break

      ready, _, _ = select.select([proc.stdout, proc.stderr], [], [], 0.2)
      for stream in ready:
          line = stream.readline()
          if not line:
              continue
          line = line.rstrip("\n")
          if stream is proc.stdout:
              out_lines.append(line)
              print(f"[val][out] {line}")
          else:
              err_lines.append(line)
              print(f"[val][err] {line}")

  # Drain remaining output
  out_rest, err_rest = proc.communicate(timeout=2)
  if out_rest:
      for line in out_rest.splitlines():
          out_lines.append(line)
          print(f"[val][out] {line}")
  if err_rest:
      for line in err_rest.splitlines():
          err_lines.append(line)
          print(f"[val][err] {line}")

  out_text = "\n".join(out_lines).strip()
  err_text = "\n".join(err_lines).strip()
  merged = (out_text + ("\n" + err_text if err_text else "")).strip()

  val_ok = "Plan valid" in merged
  print(f"[val] done. valid={val_ok}")
  return {
      "val_ok": val_ok,
      "val_output": merged,
  }

In [20]:
# Cell 8: Validate plan with VAL (terminal-equivalent capture, no timeout)
def run_val(domain_pddl_path: Path, problem_pddl_path: Path, plan_path: Path):
  if not VAL_PATH.exists():
      return {
          "val_ok": False,
          "val_output": f"VAL executable not found: {VAL_PATH}",
      }

  cmd = [str(VAL_PATH), "-v", str(domain_pddl_path), str(problem_pddl_path), str(plan_path)]
  print(f"[val] start: {' '.join(cmd)}")

  proc = subprocess.run(
      cmd,
      capture_output=True,
      text=True,
  )

  stdout = proc.stdout or ""
  stderr = proc.stderr or ""
  merged = (stdout + ("\n" + stderr if stderr else "")).strip()

  val_ok = "Plan valid" in merged
  print(f"[val] done. valid={val_ok}")
  return {
      "val_ok": val_ok,
      "val_output": merged,
  }

In [10]:
# Cell 9: Save generated artifacts for one problem
def save_problem_outputs(problem_id: str, raw_text: str, problem_pddl_text: str, solver_result: dict, val_result: dict | None):
  problem_dir = OUTPUT_ROOT / problem_id
  cand_dir = problem_dir / "cand_00"
  cand_dir.mkdir(parents=True, exist_ok=True)

  # Raw model output
  (cand_dir / "raw.txt").write_text(raw_text or "")

  # Generated problem PDDL
  (cand_dir / "problem.pddl").write_text(problem_pddl_text or "")

  # Always store fixed domain used for solve/val (for reproducibility)
  (cand_dir / "domain.pddl").write_text(DOMAIN_PDDL_TEXT)

  # Solver logs
  (cand_dir / "solver_stdout.txt").write_text(solver_result.get("stdout", "") or "")
  (cand_dir / "solver_stderr.txt").write_text(solver_result.get("stderr", "") or "")

  if solver_result.get("plan_found"):
      (cand_dir / "plan.txt").write_text(solver_result.get("plan_text", "") or "")
  else:
      (cand_dir / "solver_error.txt").write_text(solver_result.get("solver_error", "") or "")

  # VAL log (only when VAL was run)
  if val_result is not None:
      (cand_dir / "validator.log").write_text(val_result.get("val_output", "") or "")

  return cand_dir

In [11]:
# Cell 10: Run one problem through full pipeline
def process_one_problem(problem_id: str):
  prompt = build_prompt(problem_id)
  raw_text = call_gemini(prompt)

  problem_pddl_text = extract_problem_pddl(raw_text)

  # If model fails to return a tagged problem file, record and stop early
  if not problem_pddl_text.strip():
      solver_result = {
          "plan_found": False,
          "plan_text": "",
          "solver_error": "No <problem_file>...</problem_file> found in model output.",
          "stdout": "",
          "stderr": "",
      }
      save_problem_outputs(problem_id, raw_text, "", solver_result, None)
      return {
          "problem": problem_id,
          "parse_ok": False,
          "plan_found": False,
          "val_ok": False,
          "solver_error": solver_result["solver_error"],
      }

  # Write generated problem first so solver can consume it
  problem_dir = OUTPUT_ROOT / problem_id / "cand_00"
  problem_dir.mkdir(parents=True, exist_ok=True)
  gen_problem_path = problem_dir / "problem.pddl"
  gen_problem_path.write_text(problem_pddl_text)

  solver_result = run_solver(
      domain_pddl_path=DOMAIN_PDDL_PATH,
      problem_pddl_path=gen_problem_path,
      work_dir=problem_dir,
      timeout_sec=300,
  )

  val_result = None
  val_ok = False
  if solver_result["plan_found"]:
      plan_path = problem_dir / "plan"
      if not plan_path.exists():
          # fallback safety if solver wrote plan text but not file
          plan_path = problem_dir / "plan.txt"
          plan_path.write_text(solver_result["plan_text"])
      val_result = run_val(DOMAIN_PDDL_PATH, gen_problem_path, plan_path)
      val_ok = bool(val_result["val_ok"])

  # Persist final artifacts
  save_problem_outputs(problem_id, raw_text, problem_pddl_text, solver_result, val_result)

  return {
      "problem": problem_id,
      "parse_ok": True,
      "plan_found": bool(solver_result["plan_found"]),
      "val_ok": val_ok,
      "solver_error": solver_result.get("solver_error", ""),
  }

In [56]:
# Cell 11: Generate only (no solver/VAL)
smoke_problem = "p96"

prompt = build_prompt(smoke_problem)
raw_text = call_gemini(prompt)
problem_pddl_text = extract_problem_pddl(raw_text)

cand_dir = OUTPUT_ROOT / smoke_problem / "cand_00"
cand_dir.mkdir(parents=True, exist_ok=True)

(cand_dir / "raw.txt").write_text(raw_text or "")
(cand_dir / "problem.pddl").write_text(problem_pddl_text or "")
(cand_dir / "domain.pddl").write_text(DOMAIN_PDDL_TEXT)

print("Generation-only done.")
print("problem:", smoke_problem)
print("extracted_problem_pddl_chars:", len(problem_pddl_text or ""))
print("saved:", cand_dir)

Generation-only done.
problem: p96
extracted_problem_pddl_chars: 3227
saved: /home/sagemaker-user/formalizer-scaling/output/direct_pddl/gemini_3_flash/p96/cand_00


In [55]:
# Cell 12: Solver+VAL only (no model call, concise output) -- VAL uses ground-truth problem file
import contextlib
import io

smoke_problem = "p96"
cand_dir = OUTPUT_ROOT / smoke_problem / "cand_00"

problem_path = cand_dir / "problem.pddl"  # LLM-generated problem for solver
if not problem_path.exists() or not problem_path.read_text().strip():
  raise FileNotFoundError(f"Missing/empty generated problem file: {problem_path}")

gt_problem_path = DATA_DIR / "pddl" / f"{smoke_problem}.pddl"  # ground-truth problem for VAL
if not gt_problem_path.exists():
  raise FileNotFoundError(f"Missing ground-truth problem file for VAL: {gt_problem_path}")

# Suppress verbose live logs from solver in notebook output
with contextlib.redirect_stdout(io.StringIO()):
  solver_result = run_solver(
      domain_pddl_path=DOMAIN_PDDL_PATH,
      problem_pddl_path=problem_path,
      work_dir=cand_dir,
      timeout_sec=300,
  )

# Save solver logs/results
(cand_dir / "solver_stdout.txt").write_text(solver_result.get("stdout", "") or "")
(cand_dir / "solver_stderr.txt").write_text(solver_result.get("stderr", "") or "")

produced_files = ["solver_stdout.txt", "solver_stderr.txt"]
val_result = None

if solver_result["plan_found"]:
  # keep a canonical copy as plan.txt and use it for VAL
  plan_file = cand_dir / "plan.txt"
  plan_text = solver_result.get("plan_text", "")
  (cand_dir / "plan.txt").write_text(plan_text)
  produced_files.append("plan.txt")

  # Suppress verbose live logs from VAL in notebook output
  with contextlib.redirect_stdout(io.StringIO()):
      # IMPORTANT: VAL validates against ground-truth problem file
      val_result = run_val(DOMAIN_PDDL_PATH, gt_problem_path, plan_file)

  (cand_dir / "validator.log").write_text(val_result.get("val_output", "") or "")
  produced_files.append("validator.log")
else:
  (cand_dir / "solver_error.txt").write_text(solver_result.get("solver_error", "") or "")
  produced_files.append("solver_error.txt")

summary = {
  "problem": smoke_problem,
  "plan_found": bool(solver_result["plan_found"]),
  "val_ok": bool(val_result["val_ok"]) if val_result is not None else False,
  "solver_error": solver_result.get("solver_error", ""),
}

print("Produced files:")
for name in produced_files:
  print("-", name)

if val_result is not None:
  print(f"VAL result: {'valid' if val_result.get('val_ok') else 'invalid'}")
else:
  print("VAL result: not run (solver failed)")

print("Summary:")
print(json.dumps(summary, indent=2))

Produced files:
- solver_stdout.txt
- solver_stderr.txt
- plan.txt
- validator.log
VAL result: valid
Summary:
{
  "problem": "p95",
  "plan_found": true,
  "val_ok": true,
  "solver_error": ""
}


In [ ]:
# Cell 13: Batch generation-only (with tqdm), default p102..p110
from tqdm.auto import tqdm

BATCH_PROBLEMS = [f"p{i:03d}" for i in range(191, 201)]  # change this list for other batches

gen_results = []
for pid in tqdm(BATCH_PROBLEMS, desc="LLM generation", unit="problem"):
  try:
      prompt = build_prompt(pid)
      raw_text = call_gemini(prompt)
      problem_pddl_text = extract_problem_pddl(raw_text)

      cand_dir = OUTPUT_ROOT / pid / "cand_00"
      cand_dir.mkdir(parents=True, exist_ok=True)

      (cand_dir / "raw.txt").write_text(raw_text or "")
      (cand_dir / "problem.pddl").write_text(problem_pddl_text or "")
      (cand_dir / "domain.pddl").write_text(DOMAIN_PDDL_TEXT)

      ok = bool(problem_pddl_text.strip())
      gen_results.append({"problem": pid, "generation_ok": ok, "error": ""})
      print(f"{pid}: generation {'ok' if ok else 'failed (no <problem_file>)'}")
  except Exception as e:
      gen_results.append({"problem": pid, "generation_ok": False, "error": str(e)})
      print(f"{pid}: generation error -> {e}")

print("\nGeneration batch done.")
print("total:", len(gen_results))
print("ok:", sum(1 for r in gen_results if r['generation_ok']))
print("failed:", sum(1 for r in gen_results if not r['generation_ok']))

In [ ]:
# Cell 14: Batch solver+VAL only (self-contained, with its own problem handle)
from tqdm.auto import tqdm
import contextlib
import io

BATCH_PROBLEMS_SOLVER_VAL = [f"p{i:03d}" for i in range(102, 111)]  # change as needed

sv_results = []
for pid in tqdm(BATCH_PROBLEMS_SOLVER_VAL, desc="Solver+VAL", unit="problem"):
  cand_dir = OUTPUT_ROOT / pid / "cand_00"
  problem_path = cand_dir / "problem.pddl"

  if not problem_path.exists() or not problem_path.read_text().strip():
      msg = f"missing/empty problem file: {problem_path}"
      sv_results.append({"problem": pid, "plan_found": False, "val_ok": False, "error": msg})
      print(f"{pid}: skipped ({msg})")
      continue

  # Suppress verbose live logs in batch mode
  with contextlib.redirect_stdout(io.StringIO()):
      solver_result = run_solver(
          domain_pddl_path=DOMAIN_PDDL_PATH,
          problem_pddl_path=problem_path,
          work_dir=cand_dir,
          timeout_sec=300,
      )

  (cand_dir / "solver_stdout.txt").write_text(solver_result.get("stdout", "") or "")
  (cand_dir / "solver_stderr.txt").write_text(solver_result.get("stderr", "") or "")

  val_ok = False
  if solver_result.get("plan_found"):
      plan_file = cand_dir / "plan"
      (cand_dir / "plan.txt").write_text(solver_result.get("plan_text", "") or "")

      with contextlib.redirect_stdout(io.StringIO()):
          val_result = run_val(DOMAIN_PDDL_PATH, problem_path, plan_file)

      (cand_dir / "validator.log").write_text(val_result.get("val_output", "") or "")
      val_ok = bool(val_result.get("val_ok", False))
      print(f"{pid}: val completed ({'valid' if val_ok else 'invalid'})")
  else:
      (cand_dir / "solver_error.txt").write_text(solver_result.get("solver_error", "") or "")
      print(f"{pid}: solver failed")

  sv_results.append({
      "problem": pid,
      "plan_found": bool(solver_result.get("plan_found", False)),
      "val_ok": val_ok,
      "error": solver_result.get("solver_error", ""),
  })

print("\nSolver+VAL batch done.")
print("total:", len(sv_results))
print("plan_found:", sum(1 for r in sv_results if r["plan_found"]))
print("val_ok:", sum(1 for r in sv_results if r["val_ok"]))
print("failed:", sum(1 for r in sv_results if not r["val_ok"]))

# 1b. Gemini-3-Flash-Preview (fast downward)

In [69]:
# Cell 1 (Fast Downward): Setup imports, paths, and Gemini key
from pathlib import Path
import os
import re
import json
import time
import subprocess
from datetime import datetime

from dotenv import load_dotenv
from google import genai

# Project root (run notebook from formalizer-scaling/src/pddl_gen)
PROJECT_ROOT = Path.cwd().parents[1]

DATA_DIR = PROJECT_ROOT / "data"
PROMPT_PATH = PROJECT_ROOT / "prompts" / "only_pddl_instruction.txt"
DOMAIN_PDDL_PATH = DATA_DIR / "pddl" / "domain.pddl"
DESC_DIR = DATA_DIR / "descriptions"

# Keep same output folder as existing Gemini runs
OUTPUT_ROOT = PROJECT_ROOT / "output" / "direct_pddl" / "gemini_3_flash"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

VAL_PATH = Path("/home/sagemaker-user/tools/VAL/build/linux64/Release/bin/Validate")
FD_PATH = Path("/home/sagemaker-user/tools/downward/fast-downward.py")

# Load .env and configure Gemini client
load_dotenv(PROJECT_ROOT / ".env")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
if not GOOGLE_API_KEY:
  raise ValueError("GOOGLE_API_KEY not found in .env")

client = genai.Client(api_key=GOOGLE_API_KEY)

# Gemini model code
MODEL_NAME = "gemini-3-flash-preview"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("VAL exists:", VAL_PATH.exists())
print("Fast Downward exists:", FD_PATH.exists())
print("Domain PDDL exists:", DOMAIN_PDDL_PATH.exists())
print("Prompt exists:", PROMPT_PATH.exists())
print("Model:", MODEL_NAME)

PROJECT_ROOT: /home/sagemaker-user/formalizer-scaling
OUTPUT_ROOT: /home/sagemaker-user/formalizer-scaling/output/direct_pddl/gemini_3_flash
VAL exists: True
Fast Downward exists: True
Domain PDDL exists: True
Prompt exists: True
Model: gemini-3-flash-preview


In [73]:
# Cell 2 (Fast Downward): Define target problems and load static prompt/domain text
PROBLEM_IDS = [f"p{i:03d}" for i in range(101, 201)]  # p101..p200

PROMPT_PREFIX = PROMPT_PATH.read_text()
DOMAIN_PDDL_TEXT = DOMAIN_PDDL_PATH.read_text()

print("Num problems:", len(PROBLEM_IDS))
print("First/last:", PROBLEM_IDS[0], PROBLEM_IDS[-1])
print("Prompt chars:", len(PROMPT_PREFIX))
print("Domain PDDL chars:", len(DOMAIN_PDDL_TEXT))

Num problems: 100
First/last: p101 p200
Prompt chars: 2544
Domain PDDL chars: 997


In [74]:
# Cell 3 (Fast Downward): Load NL descriptions for one problem ID
def load_problem_descriptions(problem_id: str):
  domain_path = DESC_DIR / f"{problem_id}_domain.txt"
  problem_path = DESC_DIR / f"{problem_id}_problem.txt"

  if not domain_path.exists():
      raise FileNotFoundError(f"Missing domain description: {domain_path}")
  if not problem_path.exists():
      raise FileNotFoundError(f"Missing problem description: {problem_path}")

  return domain_path.read_text(), problem_path.read_text()

In [75]:
# Cell 4 (Fast Downward): Build prompt for one problem (combined primer + 3-input setup)
def build_prompt(problem_id: str):
  nl_domain, nl_problem = load_problem_descriptions(problem_id)

  return (
      f"{PROMPT_PREFIX}\n\n"
      "You are an expert PDDL engineer.\n"
      "Create a syntactically correct and semantically consistent problem file "
      "using the information below.\n\n"
      "You are given three inputs:\n"
      "1) Original natural language domain description\n"
      "2) Original natural language problem description\n"
      "3) Ground-truth domain PDDL (fixed reference)\n\n"
      "### Original domain description\n"
      f"{nl_domain}\n\n"
      "### Original problem description\n"
      f"{nl_problem}\n\n"
      "### Ground-truth domain PDDL\n"
      f"{DOMAIN_PDDL_TEXT}\n\n"
      "Output contract:\n"
      "- Generate ONLY the final problem PDDL.\n"
      "- Wrap it in exactly one <problem_file>...</problem_file> block.\n"
      "- Do not output any <domain_file> block.\n"
      "- Do not include any extra text before or after <problem_file>...</problem_file>."
  )

In [76]:
# Cell 5 (Fast Downward): Call Gemini for one prompt
def call_gemini(prompt: str, temperature: float = 0.4, max_output_tokens: int = 150000):
  response = client.models.generate_content(
      model=MODEL_NAME,
      contents=prompt,
      config={
          "temperature": temperature,
          "max_output_tokens": max_output_tokens,
      },
  )
  return (response.text or "").strip()

In [77]:
# Cell 6 (Fast Downward): Parse model output and extract problem_file block
def extract_problem_pddl(raw_text: str):
  if not raw_text:
      return ""

  matches = re.findall(r"<problem_file>(.*?)</problem_file>", raw_text, flags=re.DOTALL | re.IGNORECASE)
  if matches:
      return matches[-1].strip()

  return ""

In [78]:
# Cell 7 (Fast Downward): Solve one generated problem locally with Fast Downward
def run_solver(domain_pddl_path: Path, problem_pddl_path: Path, work_dir: Path, timeout_sec: int = 300):
  if not FD_PATH.exists():
      return {
          "plan_found": False,
          "plan_text": "",
          "solver_error": f"Fast Downward not found: {FD_PATH}",
          "stdout": "",
          "stderr": "",
      }

  # Fast Downward writes plan to sas_plan by default
  plan_path = work_dir / "sas_plan"
  if plan_path.exists():
      plan_path.unlink()

  cmd = [
      str(FD_PATH),
      str(domain_pddl_path),
      str(problem_pddl_path),
      "--search",
      "astar(lmcut())",
  ]
  print(f"[fd] start: {' '.join(cmd)}")
  print(f"[fd] timeout: {timeout_sec}s")

  try:
      proc = subprocess.run(
          cmd,
          cwd=work_dir,
          capture_output=True,
          text=True,
          timeout=timeout_sec,
      )
      stdout = proc.stdout or ""
      stderr = proc.stderr or ""
  except subprocess.TimeoutExpired as e:
      return {
          "plan_found": False,
          "plan_text": "",
          "solver_error": f"timeout after {timeout_sec}s",
          "stdout": e.stdout or "",
          "stderr": e.stderr or "",
      }

  # Fast Downward can emit multiple plan files for anytime configs; here expect sas_plan
  if plan_path.exists():
      plan_text = plan_path.read_text().strip()
      if plan_text:
          # Remove optional comment lines for VAL compatibility clarity
          plan_lines = [ln for ln in plan_text.splitlines() if not ln.strip().startswith(";")]
          cleaned = "\n".join(plan_lines).strip()
          return {
              "plan_found": bool(cleaned),
              "plan_text": cleaned,
              "solver_error": "" if cleaned else "empty plan after stripping comments",
              "stdout": stdout,
              "stderr": stderr,
          }

  err = (stderr + "\n" + stdout).strip() or "plan not found"
  return {
      "plan_found": False,
      "plan_text": "",
      "solver_error": err,
      "stdout": stdout,
      "stderr": stderr,
  }

In [79]:
# Cell 8 (Fast Downward): Validate plan with VAL (live logs, no timeout)
import select

def run_val(domain_pddl_path: Path, problem_pddl_path: Path, plan_path: Path):
  if not VAL_PATH.exists():
      return {
          "val_ok": False,
          "val_output": f"VAL executable not found: {VAL_PATH}",
      }

  cmd = [str(VAL_PATH), "-v", str(domain_pddl_path), str(problem_pddl_path), str(plan_path)]
  print(f"[val] start: {' '.join(cmd)}")

  proc = subprocess.Popen(
      cmd,
      stdout=subprocess.PIPE,
      stderr=subprocess.PIPE,
      text=True,
      bufsize=1,
  )

  out_lines = []
  err_lines = []

  # Live-stream until process exits
  while True:
      if proc.poll() is not None:
          break

      ready, _, _ = select.select([proc.stdout, proc.stderr], [], [], 0.2)
      for stream in ready:
          line = stream.readline()
          if not line:
              continue
          line = line.rstrip("\n")
          if stream is proc.stdout:
              out_lines.append(line)
              print(f"[val][out] {line}")
          else:
              err_lines.append(line)
              print(f"[val][err] {line}")

  # Drain remaining output
  out_rest, err_rest = proc.communicate(timeout=2)
  if out_rest:
      for line in out_rest.splitlines():
          out_lines.append(line)
          print(f"[val][out] {line}")
  if err_rest:
      for line in err_rest.splitlines():
          err_lines.append(line)
          print(f"[val][err] {line}")

  out_text = "\n".join(out_lines).strip()
  err_text = "\n".join(err_lines).strip()
  merged = (out_text + ("\n" + err_text if err_text else "")).strip()

  val_ok = "Plan valid" in merged
  print(f"[val] done. valid={val_ok}")
  return {
      "val_ok": val_ok,
      "val_output": merged,
  }

In [80]:
# Cell 9 (Fast Downward): Save generated artifacts for one problem
def save_problem_outputs(problem_id: str, raw_text: str, problem_pddl_text: str, solver_result: dict, val_result: dict | None):
  problem_dir = OUTPUT_ROOT / problem_id
  cand_dir = problem_dir / "cand_00"
  cand_dir.mkdir(parents=True, exist_ok=True)

  # Raw model output
  (cand_dir / "raw.txt").write_text(raw_text or "")

  # Generated problem PDDL
  (cand_dir / "problem.pddl").write_text(problem_pddl_text or "")

  # Always store fixed domain used for solve/val (for reproducibility)
  (cand_dir / "domain.pddl").write_text(DOMAIN_PDDL_TEXT)

  # Solver logs
  (cand_dir / "solver_stdout.txt").write_text(solver_result.get("stdout", "") or "")
  (cand_dir / "solver_stderr.txt").write_text(solver_result.get("stderr", "") or "")

  if solver_result.get("plan_found"):
      (cand_dir / "plan.txt").write_text(solver_result.get("plan_text", "") or "")
  else:
      (cand_dir / "solver_error.txt").write_text(solver_result.get("solver_error", "") or "")

  # VAL log (only when VAL was run)
  if val_result is not None:
      (cand_dir / "validator.log").write_text(val_result.get("val_output", "") or "")

  return cand_dir

In [81]:
# Cell 10 (Fast Downward): Run one problem through full pipeline
def process_one_problem(problem_id: str):
  prompt = build_prompt(problem_id)
  raw_text = call_gemini(prompt)

  problem_pddl_text = extract_problem_pddl(raw_text)

  # If model fails to return a tagged problem file, record and stop early
  if not problem_pddl_text.strip():
      solver_result = {
          "plan_found": False,
          "plan_text": "",
          "solver_error": "No <problem_file>...</problem_file> found in model output.",
          "stdout": "",
          "stderr": "",
      }
      save_problem_outputs(problem_id, raw_text, "", solver_result, None)
      return {
          "problem": problem_id,
          "parse_ok": False,
          "plan_found": False,
          "val_ok": False,
          "solver_error": solver_result["solver_error"],
      }

  # Write generated problem first so solver can consume it
  problem_dir = OUTPUT_ROOT / problem_id / "cand_00"
  problem_dir.mkdir(parents=True, exist_ok=True)
  gen_problem_path = problem_dir / "problem.pddl"
  gen_problem_path.write_text(problem_pddl_text)

  solver_result = run_solver(
      domain_pddl_path=DOMAIN_PDDL_PATH,
      problem_pddl_path=gen_problem_path,
      work_dir=problem_dir,
      timeout_sec=300,
  )

  val_result = None
  val_ok = False
  if solver_result["plan_found"]:
      # Fast Downward writes sas_plan; persist as plan.txt already done by Cell 12/14
      # Here we create/overwrite canonical plan file for VAL input.
      plan_path = problem_dir / "plan.txt"
      plan_path.write_text(solver_result["plan_text"])
      val_result = run_val(DOMAIN_PDDL_PATH, gen_problem_path, plan_path)
      val_ok = bool(val_result["val_ok"])

  # Persist final artifacts
  save_problem_outputs(problem_id, raw_text, problem_pddl_text, solver_result, val_result)

  return {
      "problem": problem_id,
      "parse_ok": True,
      "plan_found": bool(solver_result["plan_found"]),
      "val_ok": val_ok,
      "solver_error": solver_result.get("solver_error", ""),
  }

In [ ]:
# Cell 11 (Fast Downward): Generate only (no solver/VAL)
smoke_problem = "p146"

prompt = build_prompt(smoke_problem)
raw_text = call_gemini(prompt)
problem_pddl_text = extract_problem_pddl(raw_text)

cand_dir = OUTPUT_ROOT / smoke_problem / "cand_00"
cand_dir.mkdir(parents=True, exist_ok=True)

(cand_dir / "raw.txt").write_text(raw_text or "")
(cand_dir / "problem.pddl").write_text(problem_pddl_text or "")
(cand_dir / "domain.pddl").write_text(DOMAIN_PDDL_TEXT)

print("Generation-only done.")
print("problem:", smoke_problem)
print("extracted_problem_pddl_chars:", len(problem_pddl_text or ""))
print("saved:", cand_dir)

In [83]:
# Cell 12 (Fast Downward): Solver+VAL only (no model call, concise output) -- VAL uses ground-truth problem file
import contextlib
import io

smoke_problem = "p141"
cand_dir = OUTPUT_ROOT / smoke_problem / "cand_00"

problem_path = cand_dir / "problem.pddl"  # LLM-generated problem for solver
if not problem_path.exists() or not problem_path.read_text().strip():
  raise FileNotFoundError(f"Missing/empty generated problem file: {problem_path}")

gt_problem_path = DATA_DIR / "pddl" / f"{smoke_problem}.pddl"  # ground-truth problem for VAL
if not gt_problem_path.exists():
  raise FileNotFoundError(f"Missing ground-truth problem file for VAL: {gt_problem_path}")

# Suppress verbose logs from solver in notebook output
with contextlib.redirect_stdout(io.StringIO()):
  solver_result = run_solver(
      domain_pddl_path=DOMAIN_PDDL_PATH,
      problem_pddl_path=problem_path,
      work_dir=cand_dir,
      timeout_sec=300,
  )

# Save solver logs/results
(cand_dir / "solver_stdout.txt").write_text(solver_result.get("stdout", "") or "")
(cand_dir / "solver_stderr.txt").write_text(solver_result.get("stderr", "") or "")

produced_files = ["solver_stdout.txt", "solver_stderr.txt"]
val_result = None

if solver_result["plan_found"]:
  # Fast Downward returns plan_text from sas_plan; write canonical plan.txt
  plan_file = cand_dir / "plan.txt"
  plan_text = solver_result.get("plan_text", "")
  plan_file.write_text(plan_text)
  produced_files.append("plan.txt")

  # Suppress verbose logs from VAL in notebook output
  with contextlib.redirect_stdout(io.StringIO()):
      # IMPORTANT: VAL validates against ground-truth problem file
      val_result = run_val(DOMAIN_PDDL_PATH, gt_problem_path, plan_file)

  (cand_dir / "validator.log").write_text(val_result.get("val_output", "") or "")
  produced_files.append("validator.log")
else:
  (cand_dir / "solver_error.txt").write_text(solver_result.get("solver_error", "") or "")
  produced_files.append("solver_error.txt")

summary = {
  "problem": smoke_problem,
  "plan_found": bool(solver_result["plan_found"]),
  "val_ok": bool(val_result["val_ok"]) if val_result is not None else False,
  "solver_error": solver_result.get("solver_error", ""),
}

print("Produced files:")
for name in produced_files:
  print("-", name)

if val_result is not None:
  print(f"VAL result: {'valid' if val_result.get('val_ok') else 'invalid'}")
else:
  print("VAL result: not run (solver failed)")

print("Summary:")
print(json.dumps(summary, indent=2))

Produced files:
- solver_stdout.txt
- solver_stderr.txt
- solver_error.txt
VAL result: not run (solver failed)
Summary:
{
  "problem": "p141",
  "plan_found": false,
  "val_ok": false,
  "solver_error": "b'Traceback (most recent call last):\\n  File \"<frozen runpy>\", line 198, in _run_module_as_main\\n  File \"<frozen runpy>\", line 88, in _run_code\\n  File \"/home/sagemaker-user/tools/downward/builds/release/bin/translate/__main__.py\", line 37, in <module>\\n    main()\\n  File \"/home/sagemaker-user/tools/downward/builds/release/bin/translate/main.py\", line 708, in main\\n    sas_task = pddl_to_sas(task)\\n               ^^^^^^^^^^^^^^^^^\\n  File \"/home/sagemaker-user/tools/downward/builds/release/bin/translate/main.py\", line 537, in pddl_to_sas\\n    reachable_action_params) = instantiate.explore(task)\\n                               ^^^^^^^^^^^^^^^^^^^^^^^^^\\n  File \"/home/sagemaker-user/tools/downward/builds/release/bin/translate/instantiate.py\", line 116, in explore\\

# Diagnosing cells

In [20]:
import os
os.environ["PATH"] = f"/home/sagemaker-user/.local/bin:{os.environ['PATH']}"
print(os.environ["PATH"])

/home/sagemaker-user/.local/bin:/opt/conda/bin:/opt/conda/condabin:/etc/sagemaker-inference-server:/opt/conda/condabin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin


In [21]:
!which planutils
!planutils --help

/home/sagemaker-user/.local/bin/planutils
usage: planutils [-h]
                 {install,uninstall,run,remote,remote-list,check-installed,server,list,setup,upgrade,configure,show}
                 ...

positional arguments:
  {install,uninstall,run,remote,remote-list,check-installed,server,list,setup,upgrade,configure,show}
                        sub-command help
    install             install package(s) such as a planner
    uninstall           uninstall package(s)
    run                 run package
    remote              run package remotely
    remote-list         list the available remote packages
    check-installed     check if a package is installed
    server              start a server for runnable packages
    list                list the available packages
    setup               setup planutils for current user
    upgrade             upgrade all of the installed packages
    configure           configure planutils
    show                show details about a particula

In [38]:
# Quick test cell: inspect Gemini finish reason for one prompt
test_prompt = build_prompt("p195")

resp = client.models.generate_content(
  model=MODEL_NAME,
  contents=test_prompt,
  config={
      "temperature": 0.4,
      "max_output_tokens": 150000,
  },
)

print("Response text length:", len(resp.text or ""))
print("Response text preview:\n", (resp.text or "")[:500])

# Try common locations for finish reason in SDK response object
try:
  cands = getattr(resp, "candidates", None) or []
  if cands:
      print("finish_reason:", getattr(cands[0], "finish_reason", None))
      print("safety_ratings:", getattr(cands[0], "safety_ratings", None))
  else:
      print("No candidates field found.")
except Exception as e:
  print("Could not read candidates:", e)

# Optional raw dump for debugging
print("\nRaw response object:")
print(resp)

Response text length: 5838
Response text preview:
 <problem_file>
(define
	(problem blocks-100)
	(:domain blocksworld)
	(:objects
		block1 block2 block3 block4 block5 block6 block7 block8 block9 block10 block11 block12 block13 block14 block15 block16 block17 block18 block19 block20 block21 block22 block23 block24 block25 block26 block27 block28 block29 block30 block31 block32 block33 block34 block35 block36 block37 block38 block39 block40 block41 block42 block43 block44 block45 block46 block47 block48 block49 block50 block51 block52 block53 bloc
finish_reason: FinishReason.STOP
safety_ratings: None

Raw response object:
sdk_http_response=HttpResponse(
  headers=<dict len=11>
) candidates=[Candidate(
  citation_metadata=CitationMetadata(
    citations=[
      Citation(
        end_index=584,
        license='',
        start_index=176,
        uri='https://stackoverflow.com/questions/61443414/how-to-solve-programming-chess-piece-logical-problem'
      ),
      Citation(
        end_inde